# Table of Contents
- [Import Libraries](#import-libraries)
- [Data Paths](#data-paths)
- [Loading The Data](#loading-the-data)
- [Keeping only 4G Radio Generation Connection Type](#keeping-only-4g-radio-generation-connection-type)
- [Keeping only Mobile connections](#mobile-only)
- [Checking RSRP Range](#checking-rsrp-range)
- [Keeping only the RSRP in the valid range -140 to -40 dBm](#keeping-only-the-rsrp-in-the-valid-range-140-to-40-dbm)
- [Convert Timestamp to date-time in local Riyadh time](#convert-timestamp-to-date-time-in-local-riyadh-time)
- [Normalizing DeviceManufacturer](#normalize-manufacturer)
- [Removing duplicated samples](#dedupe-samples)
- [Adding new column RSRP Quality](#adding-new-column-rsrp-quality)
- [RSRP by Operator Name](#rsrp-by-operator-name)
- [RSRP by Operator](#rsrp-by-operator)
- [RSRP by Hour by Operator](#rsrp-by-hour-by-operator)
- [RSRP by Device Manufacture](#rsrp-by-device-manufacture)
- [RSRP by Mobile Data Enabled Vs Disabled](#rsrp-by-mobile-data-enabled-vs-disabled)
- [Saving the cleaned data](#saving)


<a id="import-libraries"></a>
### Import Libraries

In [1]:
import pandas as pd 

<a id="data-paths"></a>
### Data Paths

In [2]:
rsrp_path = '../RSRP.csv'

<a id="loading-the-data"></a>
### Loading The Data

In [3]:
rsrp_df = pd.read_csv(rsrp_path)

In [ ]:
rsrp_df.head(5)

In [5]:
rsrp_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2725353 entries, 0 to 2725352
Data columns (total 11 columns):
 #   Column                  Dtype  
---  ------                  -----  
 0   Timestamp               object 
 1   LocationLatitude        float64
 2   LocationLongitude       float64
 3   RadioConnectionType     object 
 4   Country                 object 
 5   RadioNetworkGeneration  object 
 6   RadioOperatorName       object 
 7   RSRP                    int64  
 8   RadioMobileDataEnabled  object 
 9   DeviceManufacturer      object 
 10  DeviceName              object 
dtypes: float64(2), int64(1), object(8)
memory usage: 228.7+ MB


- `Checking Unique Values and Counts`

In [6]:
check_unique = [
 'RadioConnectionType',
 'Country',
 'RadioNetworkGeneration',
 'RadioOperatorName',
 'RadioMobileDataEnabled']

for col in check_unique:
    print(f"Value counts in {col}:")
    print(rsrp_df[col].value_counts())
    print("-----------------------------")

Value counts in RadioConnectionType:
RadioConnectionType
Mobile     2717295
WiFi          7390
Unknown        668
Name: count, dtype: int64
-----------------------------
Value counts in Country:


Country
Saudi Arabia    2725197
Name: count, dtype: int64
-----------------------------
Value counts in RadioNetworkGeneration:
RadioNetworkGeneration
4G         2257240
3G          445246
Unknown      17361
2G            5464
WiFi            42
Name: count, dtype: int64
-----------------------------
Value counts in RadioOperatorName:
RadioOperatorName
Operator A    1423592
Operator B     688448
Operator C     613313
Name: count, dtype: int64
-----------------------------
Value counts in RadioMobileDataEnabled:


RadioMobileDataEnabled
Enabled     2724635
Disabled        718
Name: count, dtype: int64
-----------------------------


- `Checking Nulls`

In [7]:
rsrp_df.isna().sum()

Timestamp                   0
LocationLatitude            0
LocationLongitude           0
RadioConnectionType         0
Country                   156
RadioNetworkGeneration      0
RadioOperatorName           0
RSRP                        0
RadioMobileDataEnabled      0
DeviceManufacturer          0
DeviceName                  0
dtype: int64

- `Checking Duplicates`

In [8]:
print(rsrp_df.duplicated().sum())

1812


- `Dropping Duplicates` and `Country` column since the data is from One country `Saudi Arabia`

In [9]:
rsrp_df.drop_duplicates(inplace=True)

In [10]:
rsrp_df.drop(columns = ['Country'], inplace=True)

<a id="keeping-only-4g-radio-generation-connection-type"></a>
### Keeping only `4G` Radio Generation Connection Type

In [11]:
df_4g = rsrp_df[rsrp_df['RadioNetworkGeneration'] == '4G'].copy()

In [ ]:
df_4g.head(5)

<a id="mobile-only"></a>
### Keeping only `Mobile` connections
WiFi and Unknown samples say nothing about cellular coverage.

In [13]:
df_4g = df_4g[df_4g['RadioConnectionType'] == 'Mobile']
df_4g['RadioConnectionType'].value_counts()

RadioConnectionType
Mobile    2249299
Name: count, dtype: int64

<a id="checking-rsrp-range"></a>
### Checking RSRP Range 

In [14]:
print(df_4g['RSRP'].min(), df_4g['RSRP'].max())

-140 2147483647


<a id="keeping-only-the-rsrp-in-the-valid-range-140-to-40-dbm"></a>
### Keeping only the RSRP in the valid range `-140` to `-40` dBm

In [15]:
df_4g = df_4g[df_4g['RSRP'].between(-140, -40)]

In [16]:
print(df_4g['RSRP'].min(), df_4g['RSRP'].max())

-140 -44


<a id="convert-timestamp-to-date-time-in-local-riyadh-time"></a>
### Convert `Timestamp` to date-time in local Riyadh time
The raw strings carry mixed UTC offsets, so `utc=True` is required to parse them.
Converting to `Asia/Riyadh` afterwards is what makes `Hour` mean *local* hour of day.

In [17]:
df_4g['Timestamp'] = (pd.to_datetime(df_4g['Timestamp'], utc=True)
                        .dt.tz_convert('Asia/Riyadh'))

In [18]:
df_4g.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2062219 entries, 62 to 2725352
Data columns (total 10 columns):
 #   Column                  Dtype                      
---  ------                  -----                      
 0   Timestamp               datetime64[ns, Asia/Riyadh]
 1   LocationLatitude        float64                    
 2   LocationLongitude       float64                    
 3   RadioConnectionType     object                     
 4   RadioNetworkGeneration  object                     
 5   RadioOperatorName       object                     
 6   RSRP                    int64                      
 7   RadioMobileDataEnabled  object                     
 8   DeviceManufacturer      object                     
 9   DeviceName              object                     
dtypes: datetime64[ns, Asia/Riyadh](1), float64(2), int64(1), object(6)
memory usage: 173.1+ MB


In [ ]:
df_4g.head(5)

<a id="normalize-manufacturer"></a>
### Normalizing `DeviceManufacturer`
The same brand appears in several casings (`Lenovo`, `LENOVO`, `lenovo`),
which splits it into separate rows in every groupby.

In [20]:
df_4g['DeviceManufacturer'] = df_4g['DeviceManufacturer'].str.lower()
df_4g['DeviceManufacturer'].nunique()

20

<a id="dedupe-samples"></a>
### Removing duplicated samples
The same physical sample is logged once per commercial `DeviceName`
(e.g. `D728W` and `Desire 728 Dual Sim` at the identical timestamp and location),
which inflates the sample count of the affected devices.

`DeviceManufacturer` is part of the key on purpose. Without it, two *different*
handsets that happen to sample in the same second at the same rounded location
collapse into one arbitrarily-kept row, and the manufacturer that survives is
whichever pandas met first - which quietly biases the per-manufacturer chart.
This runs after the case normalisation above, so `Lenovo` and `LENOVO` are
already the same brand by the time the key is built.

In [21]:
key = ['Timestamp', 'LocationLatitude', 'LocationLongitude',
       'RadioOperatorName', 'DeviceManufacturer']

print(len(df_4g))
df_4g = df_4g.drop_duplicates(subset=key)
print(len(df_4g))

2062219


1975645


<a id="adding-new-column-rsrp-quality"></a>
### Adding new column `RSRP Quality`: 
- `Excellent`: RSRP >= -80
- `Good`: RSRP >= -90
- `Fair`: RSRP >= -100
- `Poor`: RSRP >= -110
- `Very Poor`: else

In [22]:
bins = [-float('inf'), -110, -100, -90, -80, float('inf')]
labels = ['Very Poor', 'Poor', 'Fair', 'Good', 'Excellent']

df_4g['RSRP_Quality'] = pd.cut(df_4g['RSRP'], bins=bins, labels=labels, right=False)

In [ ]:
df_4g.head(5)

<a id="rsrp-by-operator-name"></a>
### RSRP by Operator Name

In [24]:
df_4g.groupby("RadioOperatorName")["RSRP_Quality"]\
      .value_counts(normalize=True)\
      .unstack(fill_value=0)\
      .round(2)

RSRP_Quality,Very Poor,Poor,Fair,Good,Excellent
RadioOperatorName,,,,,
Operator A,0.01,0.16,0.16,0.28,0.39
Operator B,0.01,0.08,0.25,0.28,0.38
Operator C,0.00,0.04,0.16,0.28,0.51


<a id="rsrp-by-operator"></a>
### RSRP by Operator

In [25]:
df_4g.groupby("RadioOperatorName")["RSRP"].agg(
    ["count", "mean", "median", "std"]
).sort_values("median", ascending=False)

,count,mean,median,std
RadioOperatorName,,,,
Operator C,450592,-80.876627,-80.0,11.104402
Operator A,1079060,-85.473366,-83.0,12.945300
Operator B,445993,-84.646959,-85.0,11.771124


<a id="rsrp-by-hour-by-operator"></a>
### RSRP by Hour by Operator

In [26]:
df_4g['Date'] = df_4g['Timestamp'].dt.date
df_4g['Hour'] = df_4g['Timestamp'].dt.hour

In [27]:
df_4g.groupby(['Hour', 'RadioOperatorName'])['RSRP'].agg(['count', 'mean', 'median', 'std']).sort_values('Hour')

count       mean  median        std
Hour RadioOperatorName                                     
0    Operator A         22776 -85.666359   -83.0  12.908455
     Operator B          5470 -82.057038   -82.0  14.381452
     Operator C         10966 -84.814882   -85.0  12.841239
1    Operator A         29478 -81.937513   -79.0  13.868079
     Operator B          2982 -92.644869   -96.0  13.459182
...                       ...        ...     ...        ...
22   Operator B         23317 -84.650941   -85.0  11.548480
     Operator C         25536 -81.947447   -83.0  11.141688
23   Operator A         39395 -87.649499   -87.0  13.712796
     Operator B         18670 -85.283824   -86.0  11.627153
     Operator C         10459 -86.991013   -90.0  10.443389

[72 rows x 4 columns]

<a id="rsrp-by-device-manufacture"></a>
### RSRP by Device Manufacture

In [28]:
df_4g.groupby('DeviceManufacturer')['RSRP'].agg(['count', 'mean', 'median', 'std']).sort_values('median', ascending=False)

,count,mean,median,std
DeviceManufacturer,,,,
panasonic,728,-83.498626,-78.0,8.342727
leeco,293,-83.013652,-79.0,10.489712
huawei,59068,-80.396831,-80.0,11.378884
tcl,8896,-80.658049,-81.0,11.911590
xiaomi,20245,-82.857397,-82.0,12.447577
vivo,5862,-84.490959,-82.0,13.272703
realme,1501,-84.154564,-82.0,10.103733
samsung,1760350,-84.208109,-83.0,12.495443
oppo,8211,-84.908172,-85.0,9.808947


<a id="rsrp-by-mobile-data-enabled-vs-disabled"></a>
### RSRP by Mobile Data Enabled Vs Disabled

In [29]:
df_4g.groupby("RadioMobileDataEnabled")["RSRP"].agg(
    ["count", "median", "mean"]
).sort_values('count', ascending=False)

,count,median,mean
RadioMobileDataEnabled,,,
Enabled,1975636,-83.0,-84.238410
Disabled,9,-79.0,-85.111111


<a id="saving"></a>
### Saving the cleaned data
Parquet keeps the dtypes and the timezone, and reloads far faster than the CSV.

In [30]:
df_4g.to_parquet('../rsrp_clean.parquet', index=False)
df_4g.shape

(1975645, 13)